<a href="https://colab.research.google.com/github/marinazakimi/ECAA08--Grupo-06/blob/main/etapa-02-grafos/%2017_Problemas_Hamiltonianos_e_Roteamento_de_AGV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 17 — Problemas hamiltonianos e roteamento do drone

**ECAA08 · Grupo 06 · Drone agrícola de pulverização**

O nome do arquivo mantém `Roteamento_de_AGV` para correspondência com o modelo do professor. O veículo e a aplicação estudados aqui são o **drone agrícola**.


## Formulação e limite do problema

O TSP determina a ordem de visita a cinco pontos de serviço e o retorno à BASE. Um ciclo hamiltoniano visita cada vértice uma vez, exceto a repetição da base para fechar o percurso; um circuito euleriano cobre arestas. Os pontos de serviço são referências de talhões, com coordenadas locais hipotéticas em metros.

O grafo é completo, não dirigido e simétrico; $d_{ij}=\sqrt{(x_i-x_j)^2+(y_i-y_j)^2}$. Assume-se que todos os segmentos são permitidos e que o vento não modifica o custo. Se houver obstáculos, deve-se construir uma matriz de distâncias de corredores permitidos. Visitar um ponto de talhão **não representa cobrir sua área**: faixas, sobreposição e manobras internas não são calculadas pelo TSP.


## Algoritmos

1. **Vizinho mais próximo:** a partir da base, visitar sempre o ponto ainda não visitado mais próximo. Custo de construção $O(n^2)$.
2. **2-Opt:** inverter trechos da sequência e aceitar reduções no custo total, incluindo o retorno à base. A busca termina quando nenhuma troca melhora a solução; isso é um ótimo local, sem garantia geral de ótimo global.
3. **Força bruta:** fixar a base e avaliar as $5!=120$ ordens. Fornece a referência exata desta instância pequena.
4. **Prim:** construir uma árvore geradora mínima (AGM), conectando os seis pontos com cinco arestas. Seu custo é uma cota inferior para o ciclo ótimo, pois remover uma aresta do ciclo produz uma árvore geradora.

A implementação de 2-Opt recalcula cada custo em $O(n)$ e examina $O(n^2)$ candidatos: uma varredura custa $O(n^3)$. O número de varreduras depende da instância; não se afirma um limite total cúbico.


## Recursos da missão

O notebook soma deslocamento e atendimento hipotético de 30 s em cada ponto a 2 L/min. Cinco atendimentos consomem 5 L. A missão inicia com 24 L em um reservatório de capacidade 30 L, preservado da Etapa 1. O cenário usa 75% de bateria, reserva de 20% e consumo fictício de 3 pontos percentuais por minuto. A reserva de planejamento é uma hipótese adicional, não uma estimativa calibrada de autonomia real.

A aceitação exige volume suficiente acima de 1,5 L (5% de 30 L), bateria final estimada pelo menos 20% e permissivos válidos da Etapa 1. O retorno à base já integra a distância do ciclo. Se a missão não couber nos recursos, o exemplo a rejeita; não implementa divisão automática em múltiplas saídas.


## Resultados e entregável

São impressas a rota inicial, a rota melhorada, a solução exata, o desvio percentual para o ótimo, a AGM e o balanço de recursos. Os testes verificam visita única, retorno contabilizado, 2-Opt não pior, cota AGM válida e rejeição de uma missão com calda insuficiente. A igualdade entre heurística e ótimo, quando observada, vale somente para esta instância.

![Ciclo de missão e árvore geradora mínima](Arquivos_de_Apoio/Rotas_e_Arvore_Minima.png)

## Implementação executável

Python 3.10 ou superior. Execute de cima para baixo. As definições estão incluídas para permitir abrir apenas este notebook no Jupyter ou Colab. Nenhuma conexão com equipamento real é realizada.

In [1]:
from collections import deque, Counter
from math import inf, isfinite, hypot
from heapq import heappush, heappop
from itertools import permutations
from copy import deepcopy


In [2]:
def tabela(linhas):
    if not linhas:
        return '(sem registros)'
    cols = list(linhas[0])
    larg = {c: max(len(c), *(len(str(r.get(c, ''))) for r in linhas)) for c in cols}
    fmt = lambda r: ' | '.join(str(r.get(c, '')).ljust(larg[c]) for c in cols)
    return '\n'.join([fmt(dict(zip(cols, cols))), '-+-'.join('-'*larg[c] for c in cols)] + [fmt(r) for r in linhas])


In [3]:
def matriz_texto(M, linhas, colunas):
    return tabela([{'no': r, **{c: ('inf' if M[i][j] == inf else round(M[i][j], 3))
                   for j, c in enumerate(colunas)}} for i, r in enumerate(linhas)])


In [4]:
def custo_ciclo(rota, D):
    return sum(D[rota[i]][rota[(i+1)%len(rota)]] for i in range(len(rota)))


In [5]:
def vizinho_proximo(D, inicio=0):
    restantes = set(range(len(D))) - {inicio}; rota = [inicio]
    while restantes:
        v = min(restantes, key=lambda x: (D[rota[-1]][x],x))
        if not isfinite(D[rota[-1]][v]): raise ValueError('Grafo incompleto')
        restantes.remove(v); rota.append(v)
    if not isfinite(D[rota[-1]][inicio]): raise ValueError('Sem retorno')
    return rota


In [6]:
def dois_opt(rota, D):
    melhor = rota[:]
    while True:
        candidato, custo = melhor, custo_ciclo(melhor,D)
        for i in range(1,len(rota)-1):
            for j in range(i+1,len(rota)):
                nova = melhor[:i]+melhor[i:j+1][::-1]+melhor[j+1:]
                c = custo_ciclo(nova,D)
                if c < custo-1e-9: candidato,custo = nova,c
        if candidato == melhor: return melhor
        melhor = candidato


In [7]:
def tsp_exato(D):
    if len(D)>9: raise ValueError('Força bruta limitada a 9 vértices')
    rota = min(([0]+list(p) for p in permutations(range(1,len(D)))), key=lambda r:custo_ciclo(r,D))
    return rota, custo_ciclo(rota,D)


In [8]:
def prim(D):
    """Árvore geradora mínima de matriz simétrica não negativa; O(V²)."""
    n=len(D)
    if not n or any(len(r)!=n for r in D): raise ValueError('Matriz inválida')
    if any(D[i][j]<0 or D[i][j]!=D[j][i] for i in range(n) for j in range(n)):
        raise ValueError('Prim exige matriz simétrica não negativa')
    chave, pai, usados, arvore = [inf]*n, [None]*n, set(), []
    chave[0]=0
    for _ in range(n):
        u=min((v for v in range(n) if v not in usados), key=lambda v:(chave[v],v))
        if not isfinite(chave[u]): raise ValueError('Grafo desconectado')
        usados.add(u)
        if pai[u] is not None: arvore.append((pai[u],u,chave[u]))
        for v in range(n):
            if v not in usados and D[u][v]<chave[v]: chave[v],pai[v]=D[u][v],u
    return arvore, sum(w for _,_,w in arvore)


In [9]:
REGRAS = [
    ('R1', {'nivel_critico'}, 'falta_insumo'),
    ('R2', {'bomba_ligada','valvula_aberta','vazao_baixa'}, 'falha_pulverizacao'),
    ('R3', {'bomba_ligada','vazao_baixa','pressao_alta'}, 'possivel_obstrucao'),
    ('R4', {'possivel_obstrucao'}, 'interromper_pulverizacao'),
    ('R5', {'falta_insumo'}, 'interromper_pulverizacao'),
    ('R6', {'bateria_critica'}, 'interromper_missao'),
    ('R7', {'vento_alto'}, 'suspender_pulverizacao'),
    ('R8', {'bomba_ligada','vazao_baixa','pressao_baixa'}, 'possivel_vazamento_ou_falha_bomba'),
]


In [10]:
def telemetria_nominal():
    return dict(nivel_reservatorio=80.0, vazao=2.0, pressao=3.0, bateria=75.0,
                vento=3.0, bomba_ligada=True, valvula_aberta=True, gps_ok=True,
                e_stop=False, auto_mode=True, manual_mode=False, is_flying=True,
                alt_ok=True, comunicacao_ok=True)


In [11]:
def avaliar_etapa1(t):
    """Regras R1–R8 da Aula 10; fusão final ampliada nesta etapa.

    Snapshot estabilizado: não simula a dinâmica de partida da bomba.
    Dados ausentes/inválidos bloqueiam a autorização.
    """
    faixas = {'nivel_reservatorio': (0,100), 'vazao': (0,10), 'pressao': (0,6),
              'bateria': (0,100), 'vento': (0,25)}
    booleanas = ['bomba_ligada','valvula_aberta','gps_ok','e_stop','auto_mode',
                 'manual_mode','is_flying','alt_ok','comunicacao_ok']
    valido = all(isinstance(t.get(k), (int,float)) and not isinstance(t.get(k),bool)
                 and isfinite(t[k]) and lo <= t[k] <= hi for k,(lo,hi) in faixas.items())
    valido = valido and all(isinstance(t.get(k),bool) for k in booleanas)
    if not valido or not t.get('comunicacao_ok'):
        return dict(permite=False, fatos=['telemetria_invalida'], regras=[])
    f = set()
    for k in ['bomba_ligada','valvula_aberta']:
        if t[k]: f.add(k)
    conds = {'nivel_baixo':t['nivel_reservatorio']<20, 'nivel_critico':t['nivel_reservatorio']<5,
             'vazao_baixa':t['bomba_ligada'] and t['vazao']<0.5,
             'pressao_alta':t['pressao']>5,
             'pressao_baixa':t['bomba_ligada'] and t['pressao']<1,
             'bateria_baixa':t['bateria']<20, 'bateria_critica':t['bateria']<10,
             'vento_alto':t['vento']>8}
    f.update(k for k,v in conds.items() if v)
    trilha = []
    while True:
        novos = [(r,c) for r,p,c in REGRAS if p <= f and c not in f]
        if not novos: break
        for r,c in novos:
            f.add(c); trilha.append(r)
    criticos = {'falta_insumo','falha_pulverizacao','possivel_obstrucao',
                'possivel_vazamento_ou_falha_bomba','interromper_pulverizacao',
                'interromper_missao','suspender_pulverizacao'}
    global_ok = (t['gps_ok'] and t['bateria']>=20 and t['vento']<=8 and not t['e_stop']
                 and (t['auto_mode'] ^ t['manual_mode']))
    permite = (global_ok and t['is_flying'] and t['alt_ok'] and t['nivel_reservatorio']>3
                and not bool(f & criticos))
    return dict(permite=permite, fatos=sorted(f), regras=trilha)


## Experimento 1 — Distâncias, heurística e ótimo exato

In [12]:
nomes=['BASE','T1','T2','T3','T4','T5']
xy=[(0,0),(200,40),(170,100),(30,20),(60,170),(130,120)]
D=[[hypot(x-xx,y-yy) for xx,yy in xy] for x,y in xy]
print(matriz_texto(D,nomes,nomes))
inicial=vizinho_proximo(D)
melhor=dois_opt(inicial,D)
exata,otimo=tsp_exato(D)
agm,cota=prim(D)
for nome,r in [('Vizinho próximo',inicial),('2-Opt',melhor),('Ótimo exato',exata)]:
    print(nome,':',' -> '.join(nomes[i] for i in r+[r[0]]),'|',round(custo_ciclo(r,D),3),'m')
assert sorted(melhor)==list(range(6)) and melhor[0]==0
assert custo_ciclo(melhor,D)<=custo_ciclo(inicial,D)+1e-9
assert cota<=otimo+1e-9<=custo_ciclo(melhor,D)+1e-8
assert len(agm)==5
print('AGM:',[(nomes[u],nomes[v],round(w,2)) for u,v,w in agm])
print('Cota AGM:',round(cota,3),'m | Desvio para ótimo:',round(100*(custo_ciclo(melhor,D)/otimo-1),3),'%')


no   | BASE    | T1      | T2      | T3      | T4      | T5     
-----+---------+---------+---------+---------+---------+--------
BASE | 0.0     | 203.961 | 197.231 | 36.056  | 180.278 | 176.918
T1   | 203.961 | 0.0     | 67.082  | 171.172 | 191.05  | 106.301
T2   | 197.231 | 67.082  | 0.0     | 161.245 | 130.384 | 44.721 
T3   | 36.056  | 171.172 | 161.245 | 0.0     | 152.971 | 141.421
T4   | 180.278 | 191.05  | 130.384 | 152.971 | 0.0     | 86.023 
T5   | 176.918 | 106.301 | 44.721  | 141.421 | 86.023  | 0.0    
Vizinho próximo : BASE -> T3 -> T5 -> T2 -> T1 -> T4 -> BASE | 660.608 m
2-Opt : BASE -> T3 -> T1 -> T2 -> T5 -> T4 -> BASE | 585.332 m
Ótimo exato : BASE -> T3 -> T1 -> T2 -> T5 -> T4 -> BASE | 585.332 m
AGM: [('BASE', 'T3', 36.06), ('T3', 'T5', 141.42), ('T5', 'T2', 44.72), ('T2', 'T1', 67.08), ('T5', 'T4', 86.02)]
Cota AGM: 375.304 m | Desvio para ótimo: 0.0 %


## Experimento 2 — Recursos, retorno e permissivos

In [13]:
def avaliar_missao(volume_inicial_L, bateria_inicial_pct):
    distancia=custo_ciclo(melhor,D)
    tempo_min=(distancia/5 + 5*30)/60
    consumo_L=2*(5*30/60)
    bateria_final=bateria_inicial_pct-3*tempo_min
    t=telemetria_nominal()
    t.update(nivel_reservatorio=100*volume_inicial_L/30,bateria=bateria_inicial_pct)
    viavel=(0<=volume_inicial_L<=30 and volume_inicial_L-consumo_L>=1.5
            and bateria_final>=20 and avaliar_etapa1(t)['permite'])
    return dict(distancia_m=round(distancia,3),tempo_min=round(tempo_min,3),
                consumo_L=consumo_L,calda_final_L=volume_inicial_L-consumo_L,
                bateria_final_pct=round(bateria_final,3),viavel=viavel)
missao=avaliar_missao(24,75)
print(tabela([missao,avaliar_missao(4,75)]))
assert missao['viavel'] and missao['consumo_L']==5 and missao['calda_final_L']==19
assert not avaliar_missao(4,75)['viavel']
assert not avaliar_missao(24,21)['viavel']


distancia_m | tempo_min | consumo_L | calda_final_L | bateria_final_pct | viavel
------------+-----------+-----------+---------------+-------------------+-------
585.332     | 4.451     | 5.0       | 19.0          | 61.647            | True  
585.332     | 4.451     | 5.0       | -1.0          | 61.647            | False 
